# Vision Transformer (ViT) on CIFAR-10 — Full Walkthrough

This notebook walks through the entire ML pipeline:
1. Data loading & EDA
2. Preprocessing & augmentation
3. ViT architecture deep-dive
4. Training
5. Evaluation & visualisation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 1. Data Loading & EDA

In [ ]:
from src.data_preprocessing import load_cifar10, plot_sample_images, plot_class_distribution, CLASS_NAMES

(x_train, y_train), (x_val, y_val), (x_test, y_test) = load_cifar10()

print(f'Train : {x_train.shape}  labels: {y_train.shape}')
print(f'Val   : {x_val.shape}  labels: {y_val.shape}')
print(f'Test  : {x_test.shape}  labels: {y_test.shape}')

In [ ]:
plot_sample_images(x_train, y_train, n_cols=10, n_rows=3)

In [ ]:
plot_class_distribution(y_train, title='Training Set — Class Distribution')

## 2. Data Augmentation Preview

In [ ]:
from src.data_preprocessing import build_augmentation_layer
import tensorflow as tf

aug = build_augmentation_layer()

sample_img = x_train[:1]  # shape (1, 32, 32, 3)

fig, axes = plt.subplots(2, 6, figsize=(14, 5), facecolor='#0f0f0f')
axes[0][0].imshow(sample_img[0])
axes[0][0].set_title('Original', color='white', fontsize=8)
axes[0][0].axis('off')

for i, ax in enumerate(axes.flat[1:]):
    aug_img = aug(tf.cast(sample_img * 255, tf.uint8), training=True)[0].numpy()
    ax.imshow(aug_img)
    ax.set_title(f'Aug {i+1}', color='white', fontsize=8)
    ax.axis('off')

plt.suptitle('Augmentation Samples', color='white')
plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
from src.vit_model import build_vit

model = build_vit(
    image_size=32, patch_size=4, num_classes=10,
    embed_dim=64, num_heads=8, num_layers=6,
    mlp_ratio=4.0, dropout=0.1
)
model.summary(line_length=85)

In [ ]:
# Quick sanity check
dummy = np.random.rand(4, 32, 32, 3).astype('float32')
out = model(dummy, training=False)
print(f'Output shape: {out.shape}')  # (4, 10)
print(f'Softmax sums: {out.numpy().sum(axis=-1)}')

## 4. Training (runs full pipeline)

In [ ]:
# Uncomment to train (takes 20–40 min on CPU)
# from src.train import train
# model, history = train(epochs=60)

## 5. Evaluation

In [ ]:
# Load saved model and evaluate
# Uncomment after training
# from src.evaluate import main as run_evaluation
# run_evaluation()

## 6. Single Image Inference

After training, you can run inference on any image:

In [ ]:
import os
from tensorflow import keras
from src.data_preprocessing import CLASS_NAMES

MODEL_PATH = '../models/vit_cifar10.keras'

if os.path.exists(MODEL_PATH):
    saved_model = keras.models.load_model(MODEL_PATH)

    # Pick a random test image
    idx = np.random.randint(0, len(x_test))
    img = x_test[idx][np.newaxis, ...]  # (1, 32, 32, 3)

    probs = saved_model(img, training=False).numpy()[0]
    pred  = np.argmax(probs)

    plt.figure(figsize=(3, 3))
    plt.imshow(x_test[idx])
    plt.title(f'Pred: {CLASS_NAMES[pred]} ({probs[pred]*100:.1f}%)\nTrue: {CLASS_NAMES[y_test[idx]]}')
    plt.axis('off')
    plt.show()
else:
    print('Model not found. Run training first.')